# NPC EXPERIMENTAL BRAIN 

In [1]:
import numpy as np
from openai import OpenAI
from pydantic import BaseModel
from enum import Enum

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

LMSTUDIO_BASE_URL = os.environ["LMSTUDIO_BASE_URL"]
LM_API_TOKEN = os.environ["LM_API_TOKEN"]
MODEL = "gemma4:e2b"

In [3]:
# client = ollama.Client()
client = OpenAI(base_url=LMSTUDIO_BASE_URL, api_key=LM_API_TOKEN)

In [4]:
models = [m.id for m in client.models.list().data]
# models

# Couche de contrat

In [5]:
# Constantes

VOID        = 0
PLAYER      = 1
ENNEMY      = 2
GOLD        = 3
OBSTACLE    = 4
PNJ         = 5

SYMBOLS = {VOID: "·", PLAYER: "👤", ENNEMY: "👹", GOLD: "💰", OBSTACLE: "█"}

In [6]:
class Direction(str, Enum):
    UP = "UP"
    DOWN = "DOWN"
    LEFT = "LEFT"
    RIGHT = "RIGHT"

class PlayerDecision(BaseModel):
    direction: Direction
    # decisionDetails: str

MOVES = {
    "UP": (-1, 0),
    "DOWN": (1, 0),
    "LEFT": (0, -1),
    "RIGHT": (0, 1),
}

OPPOSITE = {"UP": "DOWN", "DOWN": "UP", "LEFT": "RIGHT", "RIGHT": "LEFT"}

# Moteur de perception

In [7]:
def localize(world_map, entity):
    positions = np.argwhere(world_map == entity)
    return positions

In [8]:
def compute_distance(entities_positions, pos_reference):
    if len(entities_positions) == 0:
        return np.array([])
    
    v = entities_positions - pos_reference
    distances = np.linalg.norm(v, axis=1)

    return np.round(distances, 2) # /!\ approx. distance

In [9]:
def adjacent_cells(world_map, player_position):
    rows, cols = world_map.shape
    r, c = player_position
    adjacent = {}
    for name, (d_row, d_column) in MOVES.items():
        new_row, new_column = r + d_row, c + d_column
        if not (0 <= new_row < rows and 0 <= new_column < cols):
            adjacent[name] = "WALL"
        else:
            adjacent[name] = SYMBOLS.get(world_map[new_row, new_column], "?")
    return adjacent

In [10]:
def to_direction(delta_row, delta_column):
    dirs = []
    if delta_row < 0: dirs.append("UP")
    if delta_row > 0: dirs.append("DOWN")
    if delta_column < 0: dirs.append("LEFT")
    if delta_column > 0: dirs.append("RIGHT")
    return dirs

In [11]:
def show_map(world_map):
    for row in world_map:
        print("\t".join(SYMBOLS.get(cell, "?") for cell in row))

# Moteur de déplacement

In [12]:
def allowed_move(world_map: np.ndarray, pos):
    n_rows, n_cols = world_map.shape
    r, c = pos

    if r < 0 or c < 0 or r >= n_rows or c >= n_cols:
        return False
    
    return world_map[r, c] in (VOID, GOLD)

In [13]:
def move_failure_reason(world_map: np.ndarray, pos):
    n_rows, n_cols = world_map.shape
    r, c = pos

    if r < 0 or c < 0 or r >= n_rows or c >= n_cols:
        return "EDGE_OF_MAP"

    cell = world_map[r, c]
    if cell == ENNEMY:
        return "ENEMY"
    if cell == OBSTACLE:
        return "OBSTACLE"
    if cell == PNJ:
        return "PNJ"

    return None

In [14]:
from collections import deque

def danger_cells(world_map):
    """Cases dangereuses = ennemis + leurs cases adjacentes"""
    danger = set()
    for pos in localize(world_map, ENNEMY):
        r, c = pos
        danger.add((r, c))
        for dr, dc in MOVES.values():
            danger.add((r + dr, c + dc))
    return danger

def shortest_safe_path_direction(world_map):
    """BFS vers l'or le plus proche, en évitant obstacles + zones dangereuses"""
    player_pos = tuple(localize(world_map, PLAYER)[0])
    gold_positions = {tuple(p) for p in localize(world_map, GOLD)}
    danger = danger_cells(world_map)
    n_rows, n_cols = world_map.shape

    queue = deque([(player_pos, None)])  # (position, first_direction)
    visited = {player_pos}

    while queue:
        pos, first_dir = queue.popleft()

        if pos in gold_positions:
            return first_dir

        for name, (dr, dc) in MOVES.items():
            new_pos = (pos[0] + dr, pos[1] + dc)

            if not (0 <= new_pos[0] < n_rows and 0 <= new_pos[1] < n_cols):
                continue
            if new_pos in visited:
                continue
            if world_map[new_pos] == OBSTACLE:
                continue
            if new_pos in danger and new_pos not in gold_positions:
                continue

            visited.add(new_pos)
            queue.append((new_pos, first_dir or name))

    return None  # aucun chemin sûr trouvé

In [15]:
def perception(world_map, last_move_failure=None):
    player_position = localize(world_map, PLAYER)[0]
    gold_positions = localize(world_map, GOLD)
    ennemy_positions = localize(world_map, ENNEMY)

    gold_dist = compute_distance(gold_positions, player_position)
    ennemy_dist = compute_distance(ennemy_positions, player_position)

    closest_gold_direction = to_direction(*(gold_positions[np.argmin(gold_dist)] - player_position))

    return {
        "ennemy_detection_3_radius": int(np.sum(ennemy_dist <=3)) if len(ennemy_dist) > 0 else 0,
        "ennemy_distance": ennemy_dist.tolist(),
        "ennemy_count": len(ennemy_positions),
        "closest_ennemy": float(np.min(ennemy_dist)) if len(ennemy_dist) > 0 else 999,
        "gold_distance": gold_dist.tolist(),
        "closest_gold": float(np.min(gold_dist)) if len(gold_dist) > 0 else 999,
        "closest_gold_direction": closest_gold_direction,
        "adjacent": adjacent_cells(world_map, player_position),
        "last_move_failure": last_move_failure,
        "algo_suggested_direction": shortest_safe_path_direction(world_map)  # conseil, pas une obligation
    }

In [16]:
def move(world_map: np.ndarray, old_pos, new_pos):
    move_result = {
        "gold_collected": False,
        "new_pos": old_pos
    }

    if not allowed_move(world_map, new_pos):
        return move_result

    entity = world_map[old_pos[0], old_pos[1]]
    target = world_map[new_pos[0], new_pos[1]]
    world_map[old_pos[0], old_pos[1]] = VOID
    world_map[new_pos[0], new_pos[1]] = entity

    move_result["new_pos"] = new_pos

    if target == GOLD:
        move_result["gold_collected"] = True

    return move_result

# Moteur de décision

In [17]:
def decide(player_perception: dict, history: list) -> PlayerDecision | None:
    history_text = "\n".join(
        f"- Turn {h['turn']}: tried {h['direction']} → {'SUCCESS' if h['success'] else 'FAILED'}"
        for h in history
    ) if history else "No previous moves."

    failure = player_perception.get("last_move_failure")
    failure_text = (
        f"Your last move ({failure['direction']}) FAILED because of: {failure['reason']}. Do not repeat this mistake blindly."
        if failure else "Your last move succeeded (or this is your first move)."
    )

    suggestion = player_perception.get("algo_suggested_direction")
    suggestion_text = (
        f"Algorithmic hint: a safe shortest path suggests going {suggestion}, but you decide freely."
        if suggestion else "No safe path suggestion available."
    )

    prompt = f"""
    # Context
    - I am a player who wants to collect gold.

    # Objective
    - Give me the shortest path to the gold.

    # Rule
    - You must avoid enemies
    - Avoid repeating moves that already failed

    # Last move feedback
    {failure_text}

    # Algorithmic hint
    {suggestion_text}

    # Move history
    {history_text}

    # Player perception
    {player_perception} """

    print(player_perception)

    response = client.beta.chat.completions.parse(
            model    = MODEL,
            messages = [{"role": "user", "content": prompt}],
            temperature=0,
            response_format=PlayerDecision
        )

    return response.choices[0].message.parsed or None

# Game loop (simulation)

In [18]:
initial_map = np.array([
    [0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 2, 0, 3],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 3],
    [0, 0, 0, 0, 0, 0, 0], 
])

In [19]:
def game_loop(world_map: np.ndarray, max_turns = 10):
    world_map = world_map.copy()
    history = []
    last_move_failure = None

    for turn in range(max_turns):
        print(f"\n =================== [Turn {turn + 1}] ===================")

        show_map(world_map)

        player_pos = localize(world_map, PLAYER)[0]
        p = perception(world_map, last_move_failure)
        decision: PlayerDecision | None = decide(p, history)

        if decision is not None:
            print(f"\t → LLM decision: {decision.direction.value}")

            d_row, d_col = MOVES[decision.direction.value]
            new_pos = (player_pos[0] + d_row, player_pos[1] + d_col)

            if not allowed_move(world_map, new_pos):
                reason = move_failure_reason(world_map, new_pos)
                last_move_failure = {"direction": decision.direction.value, "reason": reason}
            else:
                last_move_failure = None

            move_result = move(world_map, player_pos, new_pos)

            success = tuple(move_result["new_pos"]) != tuple(player_pos)
            history.append({
                "turn": turn + 1,
                "direction": decision.direction.value,
                "success": success
            })

            if move_result["gold_collected"]:
                print("\n >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>> FOUND GOLD ! <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<< \n")
                break

            new_pos = move_result["new_pos"]

In [20]:
game_loop(world_map=initial_map, max_turns=10)


 =================== [Turn 1] ===================
·	·	·	·	·	·	·
·	👤	·	·	👹	·	💰
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
{'ennemy_detection_3_radius': 1, 'ennemy_distance': [3.0], 'ennemy_count': 1, 'closest_ennemy': 3.0, 'gold_distance': [5.0, 6.4], 'closest_gold': 5.0, 'closest_gold_direction': ['RIGHT'], 'adjacent': {'UP': '·', 'DOWN': '·', 'LEFT': '·', 'RIGHT': '·'}, 'last_move_failure': None, 'algo_suggested_direction': 'DOWN'}
	 → LLM decision: DOWN

 =================== [Turn 2] ===================
·	·	·	·	·	·	·
·	·	·	·	👹	·	💰
·	👤	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
{'ennemy_detection_3_radius': 0, 'ennemy_distance': [3.16], 'ennemy_count': 1, 'closest_ennemy': 3.16, 'gold_distance': [5.1, 5.83], 'closest_gold': 5.1, 'closest_gold_direction': ['UP', 'RIGHT'], 'adjacent': {'UP': '·', 'DOWN': '·', 'LEFT': '·', 'RIGHT': '·'}, 'last_move_failure': None, 'algo_suggested_direction': 'DOWN'}
	 → LLM decision: RIGHT

 ============